In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Plot-Style und Warnings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# DATEN LADEN (Hier anpassen für den jeweiligen Datensatz)
# ---------------------------------------------------------
FILE_PATH = r"C:\Users\golde\PycharmProjects\pred_main_project_brueggen\data\lstm ready data\uncleaned_aufschreibung_2024to2026.csv"
DELIMITER = ";"
TARGET_COL_MANUAL = None # Optional: Manuell setzen, falls automatische Erkennung fehlschlägt

# Dummy-Daten-Erzeugung (Bitte für den echten Einsatz auskommentieren und pd.read_csv nutzen!)
# print(f"Lese Rohdaten ein von: {FILE_PATH}")
# df = pd.read_csv(FILE_PATH, sep=DELIMITER, low_memory=False)

# --- Fallback für dieses Template, damit der Code sofort ausführbar ist ---
np.random.seed(42)
df = pd.DataFrame({
    'DatumNEU': pd.date_range(start='2024-01-01', periods=1000, freq='H'),
    'Zeit von': pd.date_range(start='2024-01-01', periods=1000, freq='H').strftime('%H:%M'),
    'Sollzeit/ Stück (Min)': np.random.exponential(1.5, 1000),
    'Anzahl MA': np.random.randint(1, 10, 1000),
    'Temperatur': np.random.normal(20, 5, 1000),
    'Station': np.random.choice(['A', 'B', 'C', 'D'], 1000),
    'Bemerkung': np.random.choice(['Defekt', 'Wartung', 'OK', 'Fehler 404', 'Austausch'], 1000),
    'Störfall': np.random.choice(['J', 'N'], 1000, p=[0.1, 0.9])
})
# Duplikate für Zeitfenster-Test einbauen
df = pd.concat([df, df.iloc[0:50]]).reset_index(drop=True)
# ---------------------------------------------------------

# --- FEATURE ERKENNUNG ---
# 1. Freitext-Felder identifizieren und ausschließen
text_keywords = ['bemerkung', 'notiz', 'text', 'comment', 'beschreibung', 'info']
text_cols = [c for c in df.columns if any(kw in c.lower() for kw in text_keywords)]

# 2. Datums-/Zeit-Spalten identifizieren
time_keywords = ['datum', 'date', 'zeit', 'time', 'stempel', 'timestamp']
time_cols = [c for c in df.columns if any(kw in c.lower() for kw in time_keywords) and c not in text_cols]

# 3. Numerische und Kategoriale Features separieren (ohne Text)
df_analysis = df.drop(columns=text_cols, errors='ignore')

# Datetime sicherstellen
for tc in time_cols:
    if df_analysis[tc].dtype == 'object':
        try:
            df_analysis[tc] = pd.to_datetime(df_analysis[tc])
        except:
            pass

num_features = df_analysis.select_dtypes(include=[np.number]).columns.tolist()
cat_features = df_analysis.select_dtypes(exclude=[np.number, 'datetime', 'datetimetz']).columns.tolist()

print(f"Datensatz geladen mit {df.shape[0]} Zeilen und {df.shape[1]} Spalten.")
print(f"-> Davon als Freitext ausgeschlossen: {text_cols}")
print(f"-> Erkannte Zeit-Spalten: {time_cols}")

Datensatz geladen mit 1050 Zeilen und 8 Spalten.
-> Davon als Freitext ausgeschlossen: ['Bemerkung']
-> Erkannte Zeit-Spalten: ['DatumNEU', 'Zeit von', 'Sollzeit/ Stück (Min)']
